# AgriFM, frozen encoder (1024-d embeddings)

This notebook follows `src/models/agrifm.py` and shows how the wrapper turns the shared benchmark object into the Sentinel-2 sequence expected by AgriFM.

Key properties:
- Uses Sentinel-2 bands only.
- Resamples each parcel time series to 32 frames.
- Broadcasts each parcel-level sequence into a small spatial tile before the Swin-3D encoder.
- Pools the final encoder feature map into one embedding per parcel.

## What AgriFM expects as input

| Tensor | Shape | Meaning |
| --- | --- | --- |
| `series` | `(N, 32, 10)` | normalized Sentinel-2 band sequence |
| `pixel_values` | `(N, 32, 10, tile, tile)` | broadcast spatial tile fed to the encoder |
| output embedding | `(N, 1024)` | pooled frozen representation |

The wrapper maps benchmark Sentinel-2 band names into AgriFM order and masks missing observations before normalization.

In [ ]:
import os
import sys
import importlib.util
from pathlib import Path
import numpy as np
from dataio.get_input import Benchmark, ModalitySeries, NativeSeries

REPO = Path.cwd()
while not (REPO / 'src').exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))


def load(name, rel):
    spec = importlib.util.spec_from_file_location(name, REPO / rel)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

In [ ]:
agrifm = load('agrifm_mod', 'src/models/agrifm.py')

print('AgriFM bands:', agrifm.AGRIFM_S2_BANDS)
print('frames:', agrifm.AGRIFM_NUM_FRAMES)
print('embedding dim:', agrifm.AGRIFM_EMBEDDING_DIM)
print('default weights:', agrifm.DEFAULT_WEIGHTS_PATH)

In [ ]:
rng = np.random.default_rng(7)
N, T = 4, 18


s2_bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B10', 'B11', 'B12', 'NDVI']
s1_bands = ['VV', 'VH']
climate_bands = ['temperature', 'precipitation', 'elevation', 'slope']

months = np.arange(T, dtype=np.int64) % 12
doy = np.linspace(15, 350, T).astype(np.float32)
years = np.full(T, 2021, dtype=np.int64)

s2_values = [(rng.random((T, len(s2_bands))) * 5000).astype(np.float32) for _ in range(N)]
s1_values = [rng.normal(-12, 3, size=(T, len(s1_bands))).astype(np.float32) for _ in range(N)]
climate_values = [rng.normal(0, 1, size=(T, len(climate_bands))).astype(np.float32) for _ in range(N)]

native = NativeSeries(
    s2=ModalitySeries(s2_values, [months] * N, [doy] * N, [years] * N, s2_bands),
    s1=ModalitySeries(s1_values, [months] * N, [doy] * N, [years] * N, s1_bands),
    climate=ModalitySeries(climate_values, [months] * N, [doy] * N, [years] * N, climate_bands),
)
bench = Benchmark(
    name='synthetic',
    label_kind='multiclass',
    native=native,
    labels=np.arange(N, dtype=np.int64) % 2,
    groups=np.array(['a', 'a', 'b', 'b'], dtype=object),
    latlon=np.repeat(np.array([[11.0, 79.0]], dtype=np.float32), N, axis=0),
    label_names=['zero', 'one'],
    years=np.full(N, 2021, dtype=np.int64),
)

s2_monthly, _, _, _ = bench.monthly('s2')
s1_monthly, _, _, _ = bench.monthly('s1')
climate_monthly, _, _, _ = bench.monthly('climate')
print('s2 monthly', s2_monthly.shape, bench.s2_bands)
print('s1 monthly', s1_monthly.shape, bench.s1_bands)
print('climate monthly', climate_monthly.shape, bench.climate_bands)
print('latlon', bench.latlon.shape, 'years', bench.years.shape)

## Wrapper mapping

`AgriFMModel.to_agrifm_series` performs the model-specific conversion without loading weights. This is the cheapest way to inspect the time resampling, band order, masking, and normalization path.

In [ ]:
enc = agrifm.AgriFMModel(device='cpu')
series = enc.to_agrifm_series(bench)

print('series', series.shape)
print('mean by band', np.round(series.mean(axis=(0, 1)), 3))
print('std by band', np.round(series.std(axis=(0, 1)), 3))

## Forward pass -> embedding

The real forward pass requires the downloaded AgriFM checkpoint. It is disabled by default because this model is much heavier than the conversion check above.

In [ ]:
RUN_REAL_FORWARD = False

enc = agrifm.AgriFMModel(device='cpu')
weights_path = enc._weights_path()

if not RUN_REAL_FORWARD:
    print('Set RUN_REAL_FORWARD = True to run the frozen encoder.')
elif not weights_path.exists():
    print('Skipping encode: weights not found at', weights_path)
else:
    emb = enc.encode(bench)
    print('embedding', emb.shape)
    print('first row, first 8 dims', np.round(emb[0, :8], 4))